# 3D U-Net + GAN Pipeline for Synthetic FDG-PET Generation

This notebook implements a deep learning pipeline for generating synthetic FDG-PET images from Dynamic Amyloid PET scans. The architecture combines a 3D U-Net with a Generative Adversarial Network (GAN) to progressively refine the quality of the synthetic volumes.

> **⚠️ PRIVACY & ETHICS NOTICE:**  
> This repository contains **code only**. No real patient data, DICOM files, or Protected Health Information (PHI) are included. All data processing scripts include robust, automated anonymization routines compliant with medical data privacy standards.

In [ ]:
import os
import zipfile
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut
from pydicom.uid import generate_uid

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from tqdm import tqdm

warnings.filterwarnings('ignore')
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Data Anonymization & PHI Control
Ensures that all Protected Health Information (PHI), including institution names, patient names, and IDs, are permanently removed from DICOM headers before any processing.

In [ ]:
AMYLOID_PATH = "/kaggle/input/ano-dyn-ami-fmm"
FDG_PATH = "/kaggle/input/ano-fdg"
ANON_AMYLOID_PATH = "/kaggle/working/anon_amyloid"
ANON_FDG_PATH = "/kaggle/working/anon_fdg"

def anonymize_dicom_file(in_path: str, out_path: str):
    ds = pydicom.dcmread(in_path)
    ds.PatientName = "ANONYMIZED"
    ds.PatientID = "000000"
    ds.PatientBirthDate = ""
    ds.PatientSex = ""
    ds.InstitutionName = ""
    ds.InstitutionAddress = ""
    ds.ReferringPhysicianName = ""
    ds.StudyDescription = ""
    
    ds.StudyInstanceUID = generate_uid()
    ds.SeriesInstanceUID = generate_uid()
    ds.SOPInstanceUID = generate_uid()
    ds.remove_private_tags()
    ds.save_as(out_path)

def anonymize_folder_silent(src_folder: str, dst_folder: str) -> int:
    os.makedirs(dst_folder, exist_ok=True)
    file_count = 0
    for root, _, files in os.walk(src_folder):
        for f in files:
            if f.endswith(".dcm"):
                src = os.path.join(root, f)
                dst = os.path.join(dst_folder, os.path.relpath(src, src_folder))
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                try:
                    anonymize_dicom_file(src, dst)
                    file_count += 1
                except Exception:
                    pass
    return file_count

# --- EXECUTION (Uncomment to run locally) ---
# print("Anonymizing Amyloid PET data...")
# amyloid_count = anonymize_folder_silent(AMYLOID_PATH, ANON_AMYLOID_PATH)
# print(f"Successfully anonymized {amyloid_count} Amyloid files.")
#
# print("Anonymizing FDG-PET data...")
# fdg_count = anonymize_folder_silent(FDG_PATH, ANON_FDG_PATH)
# print(f"Successfully anonymized {fdg_count} FDG files.")

## 2. Dataset & Models

In [ ]:
class PETPairDataset(Dataset):
    def __init__(self, amyloid_dir, fdg_dir):
        self.amyloid_files = sorted([os.path.join(r, f) for r, _, fs in os.walk(amyloid_dir) for f in fs if f.endswith('.dcm')])
        self.fdg_files = sorted([os.path.join(r, f) for r, _, fs in os.walk(fdg_dir) for f in fs if f.endswith('.dcm')])
        self.min_len = min(len(self.amyloid_files), len(self.fdg_files))
        
    def __len__(self):
        return self.min_len
    
    def __getitem__(self, idx):
        def load_and_normalize(path):
            dcm = pydicom.dcmread(path)
            img = apply_voi_lut(dcm.pixel_array, dcm).astype(np.float32)
            img = (img - img.min()) / (img.max() - img.min() + 1e-8)
            return torch.tensor(img).unsqueeze(0)
            
        x = load_and_normalize(self.amyloid_files[idx])
        y = load_and_normalize(self.fdg_files[idx])
        return x.permute(0, 3, 1, 2), y.permute(0, 3, 1, 2)

class DoubleConv3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1), nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1), nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class UNet3D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1):
        super().__init__()
        self.enc1 = DoubleConv3D(in_ch, 64)
        self.pool1 = nn.MaxPool3d(2)
        self.enc2 = DoubleConv3D(64, 128)
        self.pool2 = nn.MaxPool3d(2)
        self.bottleneck = DoubleConv3D(128, 256)
        self.up1 = nn.ConvTranspose3d(256, 128, 2, stride=2)
        self.dec1 = DoubleConv3D(256, 128)
        self.up2 = nn.ConvTranspose3d(128, 64, 2, stride=2)
        self.dec2 = DoubleConv3D(128, 64)
        self.out_conv = nn.Conv3d(64, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        b = self.bottleneck(self.pool2(e2))
        d1 = self.dec1(torch.cat([self.up1(b), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d1), e1], dim=1))
        return self.out_conv(d2)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(1, 64, 4, 2, 1), nn.LeakyReLU(0.2),
            nn.Conv3d(64, 128, 4, 2, 1), nn.BatchNorm3d(128), nn.LeakyReLU(0.2),
            nn.Conv3d(128, 256, 4, 2, 1), nn.BatchNorm3d(256), nn.LeakyReLU(0.2),
            nn.Conv3d(256, 1, 4, 1, 0)
        )
    def forward(self, x): return self.net(x)

## 3. Training & Evaluation

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 2
EPOCHS = 50
LR_G = 1e-4
LR_D = 1e-4

def train_gan(generator, discriminator, dataloader, epochs=50):
    criterion_gan = nn.BCEWithLogitsLoss()
    criterion_l1 = nn.L1Loss()
    opt_g = Adam(generator.parameters(), lr=LR_G, betas=(0.5, 0.999))
    opt_d = Adam(discriminator.parameters(), lr=LR_D, betas=(0.5, 0.999))
    
    generator.train()
    discriminator.train()
    
    for epoch in range(epochs):
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")
        for amyloid, fdg in pbar:
            amyloid, fdg = amyloid.to(DEVICE), fdg.to(DEVICE)
            b_size = amyloid.size(0)
            
            real_labels = torch.ones(b_size, 1, 1, 1, 1, device=DEVICE)
            fake_labels = torch.zeros(b_size, 1, 1, 1, 1, device=DEVICE)
            
            opt_d.zero_grad()
            pred_real = discriminator(fdg)
            d_loss_real = criterion_gan(pred_real, real_labels)
            
            fake_fdg = generator(amyloid)
            pred_fake = discriminator(fake_fdg.detach())
            d_loss_fake = criterion_gan(pred_fake, fake_labels)
            d_loss = (d_loss_real + d_loss_fake) * 0.5
            d_loss.backward()
            opt_d.step()
            
            opt_g.zero_grad()
            fake_fdg = generator(amyloid)
            pred_fake_g = discriminator(fake_fdg)
            g_loss_gan = criterion_gan(pred_fake_g, real_labels)
            g_loss_l1 = criterion_l1(fake_fdg, fdg) * 100
            g_loss = g_loss_gan + g_loss_l1
            g_loss.backward()
            opt_g.step()
            
            pbar.set_postfix(G_Loss=f"{g_loss.item():.4f}", D_Loss=f"{d_loss.item():.4f}")
            
    print("Training completed.")
    return generator, discriminator

def visualize_results(generator, dataloader, num_samples=4):
    generator.eval()
    with torch.no_grad():
        amyloid, fdg = next(iter(dataloader))
        amyloid, fdg = amyloid[:num_samples].to(DEVICE), fdg[:num_samples].to(DEVICE)
        pred = generator(amyloid)
        
        fig, axes = plt.subplots(3, num_samples, figsize=(15, 8))
        for i in range(num_samples):
            axes[0, i].imshow(amyloid[i, 0].cpu().numpy().squeeze(), cmap='gray')
            axes[1, i].imshow(fdg[i, 0].cpu().numpy().squeeze(), cmap='gray')
            axes[2, i].imshow(pred[i, 0].cpu().numpy().squeeze(), cmap='gray')
            for row in range(3):
                axes[row, i].axis('off')
        axes[0, 0].set_title('Input (Amyloid)')
        axes[1, 0].set_title('Ground Truth (FDG)')
        axes[2, 0].set_title('Predicted (Synthetic FDG)')
        plt.tight_layout()
        plt.show()

# --- EXECUTION ---
# dataloader = DataLoader(PETPairDataset(ANON_AMYLOID_PATH, ANON_FDG_PATH), batch_size=BATCH_SIZE, shuffle=True)
# gen = UNet3D().to(DEVICE)
# disc = Discriminator().to(DEVICE)
# gen, disc = train_gan(gen, disc, dataloader, epochs=EPOCHS)
# visualize_results(gen, dataloader)